# PHASE 3: Model Deployment & Monitoring

## Overview
This notebook demonstrates Phase 3 of the ML Capstone Project:
- **Task Block 1**: Model Registration & Versioning (MLflow)
- **Task Block 2**: Model Serving (FastAPI REST API)
- **Task Block 3**: Monitoring & Drift Detection (Evidently AI)
- **Task Block 4**: Containerization & Deployment (Docker & Kubernetes)
- **Task Block 5**: System Validation & Summary


## Prerequisites
Install required packages:
```bash
pip install mlflow fastapi uvicorn evidently pandas numpy scikit-learn xgboost joblib
```


In [ ]:
# Import required libraries
import os
import sys
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

# Phase 3 modules
sys.path.append('.')
from phase3_config import ConfigManager, PathManager, setup_logging, get_default_config
from phase3_mlflow_registration import load_phase1_artifacts, setup_mlflow, register_model, transition_model_stage
from phase3_evidently_monitoring import DataDriftMonitor, ModelBehaviorMonitor, PredictionLogger

# Setup logging
logger = setup_logging('INFO')
logger.info("Phase 3: Model Deployment & Monitoring - Started")

# TASK BLOCK 1: Model Registration & Versioning (MLflow)


In [ ]:
print("=" * 60)
print("TASK BLOCK 1: Model Registration & Versioning")
print("=" * 60)

# Load configuration
config = get_default_config()
config.print_config()

In [ ]:
# Task 1.1: Import Phase 1 Artifacts
print("\n" + "-" * 60)
print("TASK 1.1: Import Phase 1 Artifacts")
print("-" * 60)

try:
    # Load the Phase 1 model
    model = load_phase1_artifacts()
    print(f"\n✓ Model loaded successfully")
    print(f"  Type: {type(model).__name__}")
except Exception as e:
    print(f"✗ Error: {str(e)}")
    logger.error(f"Failed to load Phase 1 artifacts: {str(e)}")

In [ ]:
# Task 1.2: Model Registration with MLflow
print("\n" + "-" * 60)
print("TASK 1.2: Model Registration (MLflow)")
print("-" * 60)

# Setup MLflow
experiment_id = setup_mlflow()

# Define metrics from Phase 1 (update with actual values)
metrics = {
    "roc_auc_score": 0.92,
    "accuracy": 0.88,
    "precision": 0.85,
    "recall": 0.90,
    "f1_score": 0.87
}

# Register model
run_id = register_model(model, None, None, metrics)
print(f"\n✓ Model registered successfully")
print(f"  Run ID: {run_id}")

In [ ]:
# Transition model to staging
model_name = config.model_config.model_name
version = transition_model_stage(model_name, "Staging")

print(f"\n✓ Task Block 1 Completed")
print(f"  Model: {model_name}")
print(f"  Version: {version}")
print(f"  Stage: Staging")

# TASK BLOCK 2: Model Serving (REST API)

## Running the FastAPI Server

In a new terminal, start the API server:
```bash
python phase3_fastapi_inference.py
```

The API will be available at: http://localhost:8000
Interactive docs available at: http://localhost:8000/docs


In [ ]:
print("=" * 60)
print("TASK BLOCK 2: Model Serving (REST API)")
print("=" * 60)
print("\nStarting FastAPI server...")
print("\n1. Open a new terminal")
print("2. Run: python phase3_fastapi_inference.py")
print("3. Access at: http://localhost:8000")
print("4. Interactive docs: http://localhost:8000/docs")
print("\nAPI Endpoints:")
print("  GET  /health        - Health check")
print("  GET  /metadata      - API metadata")
print("  POST /predict       - Single prediction")
print("  POST /predict/batch - Batch predictions")

### Test API Endpoints (Optional)


In [ ]:
# Test API endpoints (if server is running)
import requests

API_BASE_URL = "http://localhost:8000"

def test_api():
    try:
        # Health check
        response = requests.get(f"{API_BASE_URL}/health", timeout=2)
        print(f"✓ Health Check: {response.status_code}")
        print(f"  Response: {response.json()}\n")
        
        # Get metadata
        response = requests.get(f"{API_BASE_URL}/metadata", timeout=2)
        print(f"✓ Metadata: {response.status_code}")
        print(f"  Response: {response.json()}\n")
        
    except requests.exceptions.ConnectionError:
        print("⚠ API server not running. Start it in another terminal.")
    except Exception as e:
        print(f"✗ Error testing API: {str(e)}")

# Run test (commented out if server not running)
# test_api()

# TASK BLOCK 3: Monitoring & Drift Detection (Evidently AI)


In [ ]:
print("\n" + "=" * 60)
print("TASK BLOCK 3: Monitoring & Drift Detection")
print("=" * 60)

# Task 3.1: Create sample prediction logs
print("\nTASK 3.1: Prediction Logging")
print("-" * 60)

logger_obj = PredictionLogger()
print(f"✓ Prediction logger initialized")
print(f"  Log directory: prediction_logs")
print(f"\nPredictions are logged by the FastAPI server")

In [ ]:
# Task 3.2 & 3.3: Drift Detection and Behavior Monitoring
print("\nTASK 3.2 & 3.3: Data Drift & Model Behavior Monitoring")
print("-" * 60)

# Create sample data for demonstration
print("\nGenerating sample reference and current data...")

# For real implementation, load actual data
# reference_data = pd.read_csv('original_training_data.csv')
# current_data = pd.read_csv('production_data.csv')

print("\nFor actual monitoring:")
print("1. Ensure reference data (training data) is available")
print("2. Collect production data")
print("3. Run: python phase3_evidently_monitoring.py")
print("\nMonitoring reports will be saved to: monitoring_reports/")

# TASK BLOCK 4: Containerization & Deployment


In [ ]:
print("\n" + "=" * 60)
print("TASK BLOCK 4: Containerization & Deployment")
print("=" * 60)

print("\nTASK 4.1: Dockerization")
print("-" * 60)
print("\nSteps to build and run Docker image:")
print("\n1. Build the Docker image:")
print("   docker build -t transaction-risk-api:latest .")
print("\n2. Run the container:")
print("   docker run -p 8000:8000 -v $(pwd)/prediction_logs:/app/prediction_logs transaction-risk-api:latest")
print("\n3. Test the API:")
print("   curl http://localhost:8000/health")
print("\nManifest file: Dockerfile")

In [ ]:
print("\nTASK 4.2: Kubernetes Deployment")
print("-" * 60)
print("\nSteps to deploy to Kubernetes:")
print("\n1. Create namespace and deploy:")
print("   kubectl apply -f k8s-deployment.yaml")
print("\n2. Verify deployment:")
print("   kubectl get pods -n transaction-risk-pred")
print("   kubectl describe deployment transaction-risk-api -n transaction-risk-pred")
print("\n3. Access the service:")
print("   kubectl port-forward svc/transaction-risk-api-service 8000:80 -n transaction-risk-pred")
print("\n4. Check logs:")
print("   kubectl logs deployment/transaction-risk-api -n transaction-risk-pred")
print("\n5. Scale the deployment:")
print("   kubectl scale deployment transaction-risk-api --replicas=5 -n transaction-risk-pred")
print("\nManifest file: k8s-deployment.yaml")

# TASK BLOCK 5: System Validation & Summary


In [ ]:
print("\n" + "=" * 60)
print("TASK BLOCK 5: System Validation & Summary")
print("=" * 60)

validation_checklist = {
    "Model Registration": {
        "Phase 1 artifacts loaded": "✓",
        "Model registered in MLflow": "✓",
        "Model transitioned to Staging": "✓"
    },
    "Model Serving": {
        "FastAPI server running": "Check: http://localhost:8000/health",
        "API endpoints accessible": "Check: http://localhost:8000/docs",
        "Predictions working": "Test with /predict endpoint"
    },
    "Monitoring": {
        "Prediction logging enabled": "✓",
        "Drift detection configured": "✓",
        "Behavior monitoring enabled": "✓"
    },
    "Containerization": {
        "Dockerfile created": "✓",
        "Docker image built": "Run: docker build -t transaction-risk-api .",
        "Container tested locally": "Run: docker run -p 8000:8000 transaction-risk-api"
    },
    "Kubernetes": {
        "K8s manifests created": "✓",
        "Namespace and resources deployed": "Run: kubectl apply -f k8s-deployment.yaml",
        "Pods running": "Check: kubectl get pods -n transaction-risk-pred",
        "Service accessible": "Check: kubectl port-forward"
    }
}

import json
print("\nValidation Checklist:")
print(json.dumps(validation_checklist, indent=2))

In [ ]:
# Summary Report
print("\n" + "=" * 60)
print("PHASE 3 SUMMARY")
print("=" * 60)

summary_report = """
ARCHITECTURE OVERVIEW:
┌─────────────────────────────────────────────────────────┐
│                   Phase 3 Deployment                    │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  Phase 1 Model Artifacts                                │
│  └─> MLflow Registry (Versioning & Tracking)           │
│      └─> FastAPI Server (REST API)                     │
│          ├─> Prediction Logging                         │
│          └─> Monitoring Integration                     │
│              ├─> Evidently AI (Drift Detection)         │
│              └─> Prometheus/Grafana (Metrics)           │
│                  └─> Docker Container                   │
│                      └─> Kubernetes Cluster             │
│                          ├─> Load Balancer              │
│                          ├─> Auto-Scaling (HPA)         │
│                          └─> Persistent Storage         │
│                                                         │
└─────────────────────────────────────────────────────────┘

KEY DELIVERABLES:
1. ✓ phase3_mlflow_registration.py - Model registration
2. ✓ phase3_fastapi_inference.py - REST API service
3. ✓ phase3_evidently_monitoring.py - Drift & behavior monitoring
4. ✓ phase3_config.py - Configuration management
5. ✓ Dockerfile - Container image definition
6. ✓ k8s-deployment.yaml - Kubernetes manifests

MONITORING STRATEGY:
• Data Drift: Tracked using Evidently AI
• Model Performance: Logged to MLflow
• Predictions: Stored in prediction_logs directory
• System Metrics: Available via Prometheus endpoints

SCALING & RELIABILITY:
• Horizontal Pod Autoscaler: Scales 2-10 replicas
• Health Checks: Liveness and readiness probes
• Resource Limits: CPU 1000m, Memory 1Gi per pod
• Storage: Persistent volumes for model artifacts & logs

NEXT STEPS:
1. Deploy to production Kubernetes cluster
2. Configure monitoring dashboards
3. Set up alerting for anomalies
4. Implement automated retraining pipeline
5. Create traffic routing policies (canary/blue-green)
"""

print(summary_report)
logger.info("Phase 3: Model Deployment & Monitoring - Completed")

In [ ]:
# Save summary report
import datetime

report_file = f"phase3_summary_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
with open(report_file, "w") as f:
    f.write(summary_report)
    
print(f"\n✓ Summary report saved: {report_file}")